In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

base_path = '/content/drive/My Drive/MIMIC/'

df = pd.read_csv(base_path + 'training_dataset.csv')

In [3]:
df.head()

,subject_id,hadm_id,icd_code,icd_description,visit_procedure_count,anchor_age,hospital_expire_flag,report_1st,report_1st_degree,report_abnormal,...,title_transluminal,title_transluminal_coronary,title_using,title_using_catheters,vital_mean_heart_rate,vital_mean_systolic_bp,vital_mean_diastolic_bp,vital_mean_spo2,vital_mean_temp,is_male
0,10000980,26913865,41071,"Subendocardial infarction, initial episode of ...",7.0,73,0.0,0.0,0.0,0.035381,...,0.186404,0.186404,0.116653,0.145704,73.636364,142.454545,83.272727,98.909091,98.133333,0
1,10000980,26913865,42823,ICD-9 Cardiovascular Condition (Code: 42823),7.0,73,0.0,0.0,0.0,0.035381,...,0.186404,0.186404,0.116653,0.145704,73.636364,142.454545,83.272727,98.909091,98.133333,0
2,10000980,26913865,41412,ICD-9 Cardiovascular Condition (Code: 41412),7.0,73,0.0,0.0,0.0,0.035381,...,0.186404,0.186404,0.116653,0.145704,73.636364,142.454545,83.272727,98.909091,98.133333,0
3,10000980,26913865,4240,Mitral valve disorders,7.0,73,0.0,0.0,0.0,0.035381,...,0.186404,0.186404,0.116653,0.145704,73.636364,142.454545,83.272727,98.909091,98.133333,0
4,10000980,26913865,4280,"Congestive heart failure, unspecified",7.0,73,0.0,0.0,0.0,0.035381,...,0.186404,0.186404,0.116653,0.145704,73.636364,142.454545,83.272727,98.909091,98.133333,0


In [4]:
from sklearn.model_selection import GroupKFold
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

In [5]:
import re  # Added regular expressions module
# 2. Separate Target and Group Identifiers
y = df['hospital_expire_flag'].astype(int)
groups = df['subject_id']

# 3. Handle Features and One-Hot Encoding
cols_to_exclude = ['subject_id', 'hadm_id', 'icd_code', 'hospital_expire_flag']
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])
X_encoded = pd.get_dummies(X_raw, columns=['icd_description'], prefix='Dx', prefix_sep='_')
X_encoded = X_encoded.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

continuous_cols = ['anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
                   'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
                   'vital_mean_spo2', 'vital_mean_temp']

In [6]:

from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_encoded, y, groups=groups))

X_train, X_test = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

In [7]:
# 5. Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

In [8]:
# Model Training is down

In [9]:
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
param_distributions = {
    'C': np.logspace(-2, 1, 10),              # Explores strong to moderate regularization
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]     # Balances feature dropping vs feature shrinking
}

# Use a single internal Group K-Fold split to evaluate parameters quickly
cv_strategy = GroupKFold(n_splits=5)

base_lr = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight='balanced',
    max_iter=200,
    random_state=42
)
print("Running fast RandomizedSearchCV over ElasticNet parameters...")
lr_search = RandomizedSearchCV(
    estimator=base_lr,
    param_distributions=param_distributions,
    n_iter=8,  # Only evaluates the top 8 random distinct combinations to save time
    scoring='roc_auc',
    cv=cv_strategy.split(X_train, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)
# Fit search on the training data
lr_search.fit(X_train, y_train)


Running fast RandomizedSearchCV over ElasticNet parameters...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


RandomizedSearchCV(cv=<generator object _BaseKFold.split at 0x79bb9e9f2350>,
                   estimator=LogisticRegression(class_weight='balanced',
                                                max_iter=200,
                                                penalty='elasticnet',
                                                random_state=42,
                                                solver='saga'),
                   n_iter=8, n_jobs=-1,
                   param_distributions={'C': array([ 0.01      ,  0.02154435,  0.04641589,  0.1       ,  0.21544347,
        0.46415888,  1.        ,  2.15443469,  4.64158883, 10.        ]),
                                        'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]},
                   random_state=42, scoring='roc_auc')

In [10]:
from sklearn.metrics import classification_report, roc_auc_score

print("     OPTIMIZATION SEARCH RESULTS COMPLETE         ")
print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f} | l1_ratio: {lr_search.best_params_['l1_ratio']:.2f}")

# 6. Evaluate the Best Parameters on the Holdout Test Set
best_model = lr_search.best_estimator_
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

test_summary = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob_tuned
})

# Collapse down to Patient-Level Metrics
patient_res = test_summary.groupby('hadm_id').mean()
preds = (patient_res['y_prob'] >= 0.5).astype(int)

print("\n--- Optimized Patient-Level Evaluation ---")
print(classification_report(patient_res['y_true'], preds))
print(f"Optimized Patient-Level ROC-AUC Score: {roc_auc_score(patient_res['y_true'], patient_res['y_prob']):.4f}")

     OPTIMIZATION SEARCH RESULTS COMPLETE         
Best Parameters Found -> C: 0.0464 | l1_ratio: 0.70

--- Optimized Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.99      0.91      0.95       686
         1.0       0.17      0.76      0.28        17

    accuracy                           0.90       703
   macro avg       0.58      0.84      0.61       703
weighted avg       0.97      0.90      0.93       703

Optimized Patient-Level ROC-AUC Score: 0.9234
